<a href="https://colab.research.google.com/github/simichakravarty16/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**
**Name:** Simi Chakravarty

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [3]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code not in RETRY_STATUSES:
                raise requests.HTTPError(f"Unexpected status code: {r.status_code}")

            if attempt == tries - 1:
                r.raise_for_status()

        except requests.RequestException:
            if attempt == tries - 1:
                raise

        delay = 0.5 * (2 ** attempt)
        time.sleep(delay)

    raise RuntimeError("Weather request failed after every attempt")

    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [14]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
weather = get_weather(start = '2026-09-05', end = '2026-10-03')
used_fallback = False

weather = weather[weather['time'].isin(['2026-09-05', '2026-10-03'])].copy()
weather.head()

,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [16]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = weather.copy()
    wx['source'] = 'open-meteo'

wx

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [17]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

wx.head()

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [21]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [ ]:
# TODO

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [ ]:
# TODO

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_your answer here_

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [23]:
# stretch
charlottesville = get_weather(start = '2026-09-05', end = '2026-10-03', lat = 38.03, lon = -78.51)
orlando = get_weather(start = '2026-09-05', end = '2026-10-03', lat = 28.54, lon = -81.38)

dates = ['2026-09-05', '2026-10-03']



charlottesville = charlottesville[
    charlottesville['time'].isin(dates)
].copy()
orlando = orlando[
    orlando['time'].isin(dates)
].copy()



charlottesville['city'] = 'Charlottesville'
orlando['city'] = 'Orlando'



weather_cities = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)
weather_cities.head()

,time,temperature_2m_max,precipitation_sum,city
0,2026-09-05,29.4,2.5,Charlottesville
1,2026-10-03,22.0,14.7,Charlottesville
2,2026-09-05,32.1,3.9,Orlando
3,2026-10-03,30.4,14.5,Orlando
